# Shailaputri: is this server behaving abnormally right now?

28 real servers (SMD), one row per minute with causal features. Hold out one machine group at a time: the model is always tested on servers it never saw. The false-alarm rate is reported beside F1, because an alert stream nobody reads is the real failure.

Shailaputri is one of the nine models of **Sati**, the investigation agent of [Pashupatastra](https://github.com/preetraval45/PASHUPATASTRA), each named for a form of Navadurga. This notebook reproduces its core result on the public dataset, with the split that keeps the test honest.

In [ ]:
import glob, os
def find(pattern):
    hits = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
    assert hits, f"{pattern} not found under /kaggle/input"
    return hits[0]
import pandas as pd, numpy as np
d = pd.read_parquet(find("minutes.parquet"))
feats = [c for c in d.columns if c not in ("machine", "group", "minute", "anomaly")]
print(d.shape, d.groupby("group")["machine"].nunique().to_dict(), "anomaly share", round(d["anomaly"].mean(), 4))

In [ ]:
import lightgbm as lgb
from sklearn.metrics import f1_score
rows = []
for held in sorted(d["group"].unique()):
    tr, te = d[d["group"] != held], d[d["group"] == held]
    m = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, verbose=-1).fit(tr[feats], tr["anomaly"])
    p_tr = m.predict_proba(tr[feats])[:, 1]
    t = max(np.arange(0.3, 0.96, 0.05), key=lambda t: f1_score(tr["anomaly"], p_tr >= t))  # chosen on training groups
    flag = m.predict_proba(te[feats])[:, 1] >= t
    rows.append({"held out": held, "F1 (per minute)": f1_score(te["anomaly"], flag),
                 "false alarms / 1000 normal minutes": 1000 * flag[te["anomaly"].values == 0].mean()})
pd.DataFrame(rows).round(3)

**Reading it:** these are per-minute scores. The project scores per anomaly *episode* with lead time, where Shailaputri reaches F1 0.556 with 4.6 false alarms per 1000 minutes, against 0.419 and 86.6 for per-machine PCA.